# 05 — Vintage tracking (downturn vs calm)

**Plain English:** The strongest story in this data: line the vintages up by
*months on book* (not calendar time) and watch how fast each cohort goes bad.
The 2007 and 2008 cohorts originated straight into the financial crisis; 2015
originated into a calm market. The cumulative default curves separate hard — the
clearest demonstration of why vintage tracking matters.

**One-line terms**
- **Months on book** — loan age in months since first payment; puts every vintage on a common clock.
- **Cumulative default rate** — share of the cohort that has reached Stage 3 (90+/credit event) by a given age.
- **Vintage** — origination-year cohort.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

In [ ]:
# First month each loan reaches default (Stage 3), by loan age ----------
defaulted = panel[panel.stage == "Stage 3"]
first_def = defaulted.groupby(["vintage", "loan_seq"])["loan_age"].min().reset_index()
loans_per_vintage = panel.groupby("vintage")["loan_seq"].nunique()

ages = range(0, 121)  # 0..120 months on book
curves = {}
for v in m.VINTAGES:
    fv = first_def[first_def.vintage == v]
    n = loans_per_vintage[v]
    cum = [(fv.loan_age <= a).sum() / n for a in ages]
    curves[v] = cum
curve_df = pd.DataFrame(curves, index=list(ages))
curve_df.index.name = "months_on_book"
curve_df.round(4).head(13)

In [ ]:
# Plot the cumulative default curves ------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for v in m.VINTAGES:
    ax.plot(curve_df.index, 100 * curve_df[v], label=f"{v} vintage", linewidth=2)
ax.set_xlabel("months on book"); ax.set_ylabel("cumulative % ever 90+/default")
ax.set_title("Cumulative default by months on book — downturn (2007/08) vs calm (2015)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(m.OUT_CHARTS / "05_vintage_default_curves.png", dpi=130)
print("saved curve chart"); plt.close(fig)

In [ ]:
# Clean table: cumulative default at selected ages ----------------------
marks = [12, 24, 36, 48, 60, 72]
snap = (curve_df.loc[curve_df.index.isin(marks)]
        .mul(100).round(2).reset_index())
snap.columns = ["months_on_book"] + [f"{v}_cum_default_pct" for v in m.VINTAGES]
snap.to_csv(m.OUT_TABLES / "05_vintage_cumulative_default.csv", index=False)
snap